In [ ]:
!pip -q install weaviate-client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 619.5/619.5 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 3.7 MB/s eta 0:00:00


In [3]:
!pip -q install langchain langchain_community langchain_core langchain-weaviate langchain-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 619.5/619.5 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 2.7 MB/s eta 0:00:00


In [ ]:
import weaviate

In [ ]:
import os
from google.colab import userdata

In [ ]:
WEAVIATE_API_KEY = userdata.get("WEAVIATE_API_KEY")
WEAVIATE_URL=""

In [ ]:
HF_TOKEN = userdata.get("HF_TOKEN")

In [ ]:
from weaviate.classes.init import Auth

client = weaviate.connect_to_weaviate_cloud(
    cluster_url = WEAVIATE_URL, auth_credentials = Auth.api_key(WEAVIATE_API_KEY),
    headers = {
      "X-HuggingFace-Api-Key": HF_TOKEN
    },
)

In [ ]:
client.is_ready()

True

## **it is mandatory to create schema which intialize the HF tokens and other things, if not weaviate will create it using openai embedding and vectorstore.**


In [ ]:
client.collections.list_all()

{'RAG': _CollectionConfigSimple(name='RAG', description='Documents for RAG', generative_config=None, properties=[_Property(name='content', description='The content of the paragraph', data_type=<DataType.TEXT: 'text'>, index_filterable=True, index_range_filters=False, index_searchable=True, nested_properties=None, tokenization=<Tokenization.WORD: 'word'>, vectorizer_config=_PropertyVectorizerConfig(skip=False, vectorize_property_name=False), vectorizer='text2vec-huggingface', vectorizer_configs=None), _Property(name='title', description="This property was generated by Weaviate's auto-schema feature on Fri Mar 13 10:41:13 2026", data_type=<DataType.TEXT: 'text'>, index_filterable=True, index_range_filters=False, index_searchable=True, nested_properties=None, tokenization=<Tokenization.WORD: 'word'>, vectorizer_config=_PropertyVectorizerConfig(skip=False, vectorize_property_name=False), vectorizer='text2vec-huggingface', vectorizer_configs=None), _Property(name='moddate', description="Thi

In [ ]:
import weaviate.classes.config as wvc

In [ ]:
client.collections.create(
    name="RAG",
    description="Documents for RAG",
    # Setting up the HuggingFace Vectorizer
    vectorizer_config=wvc.Configure.Vectorizer.text2vec_huggingface(
        model="sentence-transformers/all-MiniLM-L6-v2",
        vectorize_collection_name=False
    ),
    # Defining your properties
    properties=[
        wvc.Property(
            name="content",
            data_type=wvc.DataType.TEXT,
            description="The content of the paragraph",
            # This replaces your moduleConfig skip/vectorize settings
            vectorize_property_name=False
        ),
    ]
)

In [ ]:
client.collections.list_all()

In [ ]:
from langchain_weaviate.vectorstores import WeaviateVectorStore

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

In [ ]:
embed_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [ ]:
vectorstore = WeaviateVectorStore(
    client= client,
    index_name="RAG",
    text_key="content",
    attributes= [],
    embedding=embed_model

)

In [33]:
from langchain_classic.retrievers import WeaviateHybridSearchRetriever

In [43]:
!pip -q install -U langchain langchain-weaviate

In [87]:
# The "alpha" parameter in search_kwargs is what triggers the hybrid logic
retriever = vectorstore.as_retriever(
    search_kwargs={
        'alpha': 0.5,  # 0.5 is the balance: 50% Vector, 50% Keyword
        'k': 5         # Number of documents to return
    }
)

In [ ]:
model_name = "HuggingFaceH4/zephyr-7b-beta"

In [ ]:
!pip -q install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.2 MB/s eta 0:00:00


In [ ]:
!pip -q install accelerate

In [ ]:
import torch
from transformers import ( AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline, )
from langchain_huggingface import HuggingFacePipeline

In [ ]:
# function for loading 4-bit quantized model
def load_quantized_model(model_name: str):
    """
    model_name: Name or path of the model to be loaded.
    return: Loaded quantized model.
    """
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        low_cpu_mem_usage=True
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,
        quantization_config=bnb_config,
    )
    return model

In [ ]:
# initializing tokenizer
def initialize_tokenizer(model_name: str):
    """
    model_name: Name or path of the model for tokenizer initialization.
    return: Initialized tokenizer.
    """
    tokenizer = AutoTokenizer.from_pretrained(model_name, return_token_type_ids=False)
    tokenizer.bos_token_id = 1  # Set beginning of sentence token id
    return tokenizer

In [ ]:
tokenizer = initialize_tokenizer(model_name)

In [ ]:
model = load_quantized_model(model_name)

In [ ]:
pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    use_cache=True,
    device_map="auto",
    #max_length=2048,
    do_sample=True,
    top_k=5,
    max_new_tokens=100,
    num_return_sequences=1,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
)

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'use_cache', 'top_k', 'pad_token_id', 'do_sample', 'num_return_sequences', 'eos_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [ ]:
llm = HuggingFacePipeline(pipeline=pipeline)

In [ ]:
!pip -q install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.2/332.2 kB 10.6 MB/s eta 0:00:00


In [26]:
from langchain_community.document_loaders import PyPDFLoader

In [27]:
doc_path = '/content/RAG_with_NLP_Tasks.pdf'

In [28]:
loader = PyPDFLoader(doc_path)

In [29]:
docs = loader.load()

In [30]:
len(docs)

19

In [31]:
def clean_metadata_for_weaviate(documents):
    for doc in documents:
        # Create a new metadata dict with valid keys
        clean_metadata = {}
        for key, value in doc.metadata.items():
            # Replace invalid characters (like dots) with underscores
            new_key = key.replace(".", "_").replace("-", "_")
            clean_metadata[new_key] = value
        doc.metadata = clean_metadata
    return documents



In [32]:
# Usage:
# documents = loader.load()
# docs = text_splitter.split_documents(documents)
cleaned_docs = clean_metadata_for_weaviate(docs)


In [ ]:

# Now add to your vectorstore
vectorstore.add_documents(cleaned_docs)

In [ ]:
len(cleaned_docs)

19

In [ ]:
retriever.invoke("What is RAG token")

In [ ]:
len(retriever.invoke("What is RAG Token"))

4

In [ ]:
re = vectorstore.similarity_search_with_score(
    "What is RAG token",
    k=4)

In [ ]:
re

In [100]:
results = vectorstore.max_marginal_relevance_search(
    query="What is RAG Token",
    k=5,lambda_mult=0.5)

In [ ]:
results[0]

In [97]:
from langchain_classic.chains import RetrievalQA

In [111]:
hybrid_chain = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=retriever)

In [ ]:
hybrid_chain.invoke("What is RAG Token?")

In [102]:
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

In [103]:
from langchain_core.prompts import ChatPromptTemplate

In [104]:
system_prompt = (
    "Use the given context to answer the question. "
    "If you don't know the answer, say you don't know. "
    "Use three sentence maximum and keep the answer concise. "
    "Context: {context}"
)

In [105]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{query}"),
    ]
)

In [106]:
from langchain_core.prompts import PromptTemplate
template = """
Use the following pieces of context to answer the question at the end.
If you don't know the answer, just say that you do not have the relevant information needed to provide a verified answer, don't try to make up an answer.
When providing an answer, aim for clarity and precision. Position yourself as a knowledgeable authority on the topic, but also be mindful to explain the information in a manner that is accessible and comprehensible to those without a technical background.
Always say "Do you have any more questions pertaining to this instrument?" at the end of the answer.
{context}
Question: {question}
Helpful Answer:"""

prompt = PromptTemplate.from_template(template)

In [107]:
rag_chain = (
    {"context": retriever, "question":RunnablePassthrough()} \
    |prompt
    |llm
)

In [ ]:
response=rag_chain.invoke("what is RAG token?")

In [66]:
collection = client.collections.get("RAG")

def hybrid_ret(query):
    response = collection.query.hybrid(
        query=query,
        alpha=0.5,
        limit = 10,
        return_metadata=["score", "distance"]
    )

In [67]:
response1 = hybrid_ret("What is RAG Token?")

In [59]:
from langchain_classic.schema import Document

collection = client.collections.get("RAG")

def hybrid_retrieve(query):
    response = collection.query.hybrid(
        query=query,
        alpha=0.5,
        limit=10,
        return_metadata=["score", "distance"]
    )

    docs = []
    for obj in response.objects:
        docs.append(
            Document(
                page_content=obj.properties["content"],
                metadata={
                    "score": obj.metadata.score,
                    "distance": obj.metadata.distance
                }
            )
        )

    return docs

In [ ]:
docs = hybrid_retrieve("What is RAG Token?")

for doc in docs:
    print(doc.page_content)
    print(doc.metadata)

In [ ]:
docs[0]

In [93]:
ret1 = retriever.invoke("What is RAG Token?")

In [94]:
len(ret1)

5

In [ ]:
docs[4]

In [ ]:
ret1[4]

In [ ]:
print(hybrid_chain.invoke("What is RAG Token?"))

In [ ]:
!pip -q install langchain langchain_community

In [4]:
from langchain_classic.retrievers import contextual_compression
from langchain_classic.retrievers.document_compressors import cohere_rerank

In [ ]:
!pip install cohere

In [ ]:
compressor = cohere_rerank(api_key = "")

In [ ]:
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=retriever
    )

In [ ]:
compressed_docs = compression_retriever.get_relevant_documents(user_query)
# Print the relevant documents from using the embeddings and reranker
print(compressed_docs)

In [ ]:
hybrid_chain = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=compression_retriever
)

In [ ]:
response = hybrid_chain.invoke("What is Abstractive Question Answering?")

In [ ]:
print(response.get("result"))